# 003-pose-search: acceptance checks

# Imports

In [ ]:
import json
import os
from contextlib import contextmanager
from dataclasses import replace

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

import uavloc.matching.correlator as correlator_module
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import build_features, correlator_config
from uavloc.features.imagenet import to_batch
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.benchmark import benchmark_draws
from uavloc.matching.correlator import CorrelatorConfig, FeatureExtractor, PositionCorrelator, crop_border, decide
from uavloc.matching.pose_search import evaluate_schedule, mean_cosine_raw, score_pose, search_pose
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, set_seed

# Params

In [ ]:
EXP = "003-pose-search"
DEVICE = "mps"
SEED = 42
TORCH_THREADS = 2
BASELINES = {"001.1": "001-seg-map-correction", "002.2": "002-perfect-satellite-map"}
LABEL_BASELINE = "002.2"                  # the class-map baseline: identity checks run on its label map
VGG_BASELINE = "001.1"

N_REPLAY_DRAWS = 12
REPLAY_SAMPLE_SEED = 0                    # the same draws as PRE_EDIT_REPLAY
REPLAY_TOLERANCE_M = 1e-6
TIE_RELATIVE_GAP = 1e-4                   # raw-score gap / surface range below float32 resolution: a tie, not a change
PRE_EDIT_REPLAY = "artifacts/replay-pre-edit.csv"
FULL_REPLAY = True                        # also replay every logged draw of both baselines through the search path
PRIMARY_NOISE_M = 15.0

HEADING_RANGE_DEG = 36
HEADING_COARSE_STEP_DEG = 3
HEADING_FINE_SPAN_DEG = 3
HEADING_FINE_STEP_DEG = 1
SEARCH_SCALES = (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)

SHAPE_SCALE = 0.6
SHAPE_TOLERANCE_CELLS = 1
NOISE_IMAGE_SEED = 0

IDENTITY_FRAMES = 5
IDENTITY_NOISE_M = 5.0
N_DIRECTIONS = 8
IDENTITY_HEADING_ERROR_DEG = 20.0         # logged heading minus true heading
IDENTITY_SCALE = 0.6
IDENTITY_HEADING_TOLERANCE_DEG = 1.0
IDENTITY_POSITION_TOLERANCE_M = 1.0
IDENTITY_SCHEDULE = "greedy"             # "greedy" (registered) | "joint" (2026-10-20 amendment)
IDENTITY_FINE_TOP_K = 1                  # "joint" only: fine pass around this many coarse pairs (amendment 2: 3)
LOGGED_RUN_REPLAY_DRAWS = 6              # per logged 003 run: replay sampled draws with the current code
CHECKS_OUT = "checks.json"               # written under artifacts/

FRAMES_DATASET = "drone-2025-easy"
SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TORCH_THREADS)

In [ ]:
frames = {f.name: f for f in load_frames(FRAMES_DATASET)}
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
len(frames)

In [ ]:
baseline_configs = {run_id: next(r["config"] for r in load_runs(exp) if r["run"] == run_id) for run_id, exp in BASELINES.items()}
logged_records = {run_id: pd.read_csv(experiment_dir(exp, create=False) / "artifacts" / f"records-{run_id}.csv")
                  for run_id, exp in BASELINES.items()}
pre_edit = pd.read_csv(experiment_dir(EXP, create=False) / PRE_EDIT_REPLAY, dtype={"run": str})
{run_id: len(records) for run_id, records in logged_records.items()}

In [ ]:
search_keys = {
    "heading_offsets_deg": [float(d) for d in range(-HEADING_RANGE_DEG, HEADING_RANGE_DEG + 1, HEADING_COARSE_STEP_DEG)],
    "heading_fine_offsets_deg": [float(d) for d in range(-HEADING_FINE_SPAN_DEG, HEADING_FINE_SPAN_DEG + 1, HEADING_FINE_STEP_DEG)],
    "scales": [float(s) for s in SEARCH_SCALES],
}

In [ ]:
samples = {run_id: records.sample(N_REPLAY_DRAWS, random_state=REPLAY_SAMPLE_SEED) for run_id, records in logged_records.items()}
snapshots = {run_id: pre_edit[pre_edit["run"] == run_id].set_index("record_index").loc[rows.index] for run_id, rows in samples.items()}
{run_id: bool((snapshots[run_id]["frame"] == rows["frame"]).all()) for run_id, rows in samples.items()}

# Model

In [ ]:
features = {run_id: build_features(config, satellite, DEVICE) for run_id, config in baseline_configs.items()}
default_correlators = {run_id: PositionCorrelator(features[run_id], correlator_config(config))
                       for run_id, config in baseline_configs.items()}
{run_id: c.config.searches_pose for run_id, c in default_correlators.items()}

# Run

# (a) No regression

In [ ]:
def replay(correlator: PositionCorrelator, rows: pd.DataFrame, forced_search: bool) -> pd.DataFrame:
    """Corrected position of each logged draw: the default path, or the pose-search path with its single hypothesis."""
    out = []
    for _, row in tqdm(list(rows.iterrows()), desc="replay", leave=False):
        frame, prior = frames[row["frame"]], LatLon(row["prior_lat"], row["prior_lon"])
        surface = search_pose(correlator, frame, prior) if forced_search else correlator.surface(frame, prior)
        correction = decide(surface, correlator.config)
        out.append({"lat": correction.position.lat, "lon": correction.position.lon, "peak_z": correction.peak_z,
                    "error_m": haversine_m(correction.position, frame.position)})
    return pd.DataFrame(out, index=rows.index)

In [ ]:
def deviation_m(result: pd.DataFrame, lat: pd.Series, lon: pd.Series) -> np.ndarray:
    return np.array([haversine_m(LatLon(a, b), LatLon(c, d)) for a, b, c, d in zip(result["lat"], result["lon"], lat, lon)])

In [ ]:
class Float64Functional:
    """`torch.nn.functional` with conv2d in float64. On this M1 (torch 2.2.2), CPU float32 conv2d with these large
    kernels returns NaN on ~5-15% of calls once MPS has been used in the process; float64 does not."""

    def __getattr__(self, name: str):
        return getattr(F, name)

    @staticmethod
    def conv2d(x: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
        return F.conv2d(x.double(), weight.double()).float()

In [ ]:
@contextmanager
def float64_conv():
    """The default correlator path with the conv flake removed; the pre-edit snapshot was taken this way."""
    correlator_module.F = Float64Functional()
    try:
        yield
    finally:
        correlator_module.F = F

In [ ]:
replays = {}
for run_id, rows in samples.items():
    replays[run_id, "default"] = replay(default_correlators[run_id], rows, forced_search=False)
    with float64_conv():
        replays[run_id, "default, float64 conv"] = replay(default_correlators[run_id], rows, forced_search=False)
    replays[run_id, "search path, one hypothesis"] = replay(default_correlators[run_id], rows, forced_search=True)

In [ ]:
replay_rows = []
for (run_id, path), result in replays.items():
    rows, snapshot = samples[run_id], snapshots[run_id]
    vs_logged = deviation_m(result, rows["corrected_lat"], rows["corrected_lon"])
    vs_pre_edit = deviation_m(result, snapshot["pre_edit_lat"], snapshot["pre_edit_lon"])
    replay_rows.append({"baseline": run_id, "path": path, "draws": len(rows),
                        "logged_within_tol": int((vs_logged <= REPLAY_TOLERANCE_M).sum()), "logged_max_m": vs_logged.max(),
                        "pre_edit_within_tol": int((vs_pre_edit <= REPLAY_TOLERANCE_M).sum()), "pre_edit_max_m": vs_pre_edit.max()})
for run_id, snapshot in snapshots.items():
    drift = snapshot["pre_edit_vs_logged_m"].to_numpy()
    replay_rows.append({"baseline": run_id, "path": "pre-edit code (snapshot)", "draws": len(drift),
                        "logged_within_tol": int((drift <= REPLAY_TOLERANCE_M).sum()), "logged_max_m": drift.max()})
replay_table = pd.DataFrame(replay_rows)
replay_table

In [ ]:
def relative_tie_gap(correlator: PositionCorrelator, row: pd.Series, reference: LatLon) -> float:
    """(raw peak - raw score at the cell `reference` sits on) / raw range, on the logged pose's float64 surface."""
    frame, prior = frames[row["frame"]], LatLon(row["prior_lat"], row["prior_lon"])
    hypothesis = score_pose(correlator, frame, prior, 0.0, 1.0)
    cell = tuple(np.round(hypothesis.surface.cell_of_offset(PositionCorrelator.patch_offset_m(prior, reference, frame.yaw_deg))).astype(int))
    raw = hypothesis.raw
    return float((raw.max() - raw[cell]) / (raw.max() - raw.min()))

In [ ]:
tie_rows = []
for run_id, rows in samples.items():
    result, snapshot = replays[run_id, "search path, one hypothesis"], snapshots[run_id]
    moved = deviation_m(result, snapshot["pre_edit_lat"], snapshot["pre_edit_lon"]) > REPLAY_TOLERANCE_M
    for index in rows.index[moved]:
        gap = relative_tie_gap(default_correlators[run_id], rows.loc[index], LatLon(snapshot.loc[index, "pre_edit_lat"], snapshot.loc[index, "pre_edit_lon"]))
        tie_rows.append({"baseline": run_id, "record_index": index, "frame": rows.loc[index, "frame"], "relative_gap": gap,
                         "tie": gap < TIE_RELATIVE_GAP})
ties = pd.DataFrame(tie_rows, columns=["baseline", "record_index", "frame", "relative_gap", "tie"])
ties

In [ ]:
default_unchanged = all(row["pre_edit_within_tol"] == row["draws"] for _, row in replay_table.iterrows()
                        if row["path"] == "default, float64 conv")
search_path_equal = bool(ties["tie"].all())
no_regression = default_unchanged and search_path_equal
default_unchanged, search_path_equal

In [ ]:
full_rows = []
if FULL_REPLAY:
    for run_id, records in logged_records.items():
        result = replay(default_correlators[run_id], records, forced_search=True)
        deviation = deviation_m(result, records["corrected_lat"], records["corrected_lon"])
        primary = records["noise_m"] == PRIMARY_NOISE_M
        full_rows.append({"baseline": run_id, "draws": len(records),
                          "within_tol_of_logged": int((deviation <= REPLAY_TOLERANCE_M).sum()),
                          "logged_mean_error_m": float(records.loc[primary, "error_m"].mean()),
                          "this_machine_mean_error_m": float(result.loc[primary, "error_m"].mean())})
full_replay = pd.DataFrame(full_rows)
full_replay

# Logged 003 runs replay

In [ ]:
logged_rows = []
for run in load_runs(EXP):
    config = run["config"]
    correlator = PositionCorrelator(build_features(config, satellite, DEVICE), correlator_config(config))
    records = pd.read_csv(experiment_dir(EXP, create=False) / "artifacts" / f"records-{run['run']}.csv")
    rows = records.sample(LOGGED_RUN_REPLAY_DRAWS, random_state=REPLAY_SAMPLE_SEED)
    deviation = deviation_m(replay(correlator, rows, forced_search=False), rows["corrected_lat"], rows["corrected_lon"])
    logged_rows.append({"run": run["run"], "registered_run": str(config.get("registered_run")), "draws": len(rows),
                        "within_tol": int((deviation <= REPLAY_TOLERANCE_M).sum()), "max_m": float(deviation.max())})
logged_replay = pd.DataFrame(logged_rows, columns=["run", "registered_run", "draws", "within_tol", "max_m"])
logged_runs_replay_ok = bool((logged_replay["within_tol"] == logged_replay["draws"]).all())
logged_replay, logged_runs_replay_ok

# Drone map at a scaled footprint

In [ ]:
shape_frame = frames[samples[LABEL_BASELINE]["frame"].iloc[0]]
shape_rows = []
for run_id, correlator in default_correlators.items():
    full_hw = tuple(correlator.drone_map(shape_frame, 1.0).shape[2:])
    scaled_hw = tuple(correlator.drone_map(shape_frame, SHAPE_SCALE).shape[2:])
    shape_rows.append({"baseline": run_id, "scale_1_hw": full_hw, f"scale_{SHAPE_SCALE:g}_hw": scaled_hw,
                       "expected_hw": tuple(round(SHAPE_SCALE * n, 2) for n in full_hw),
                       "ok": all(abs(s - SHAPE_SCALE * n) <= SHAPE_TOLERANCE_CELLS for s, n in zip(scaled_hw, full_hw))})
shapes = pd.DataFrame(shape_rows)
shapes

# Mean cosine against a noise satellite

In [ ]:
vgg = features[VGG_BASELINE]
noise_rgb = np.random.default_rng(NOISE_IMAGE_SEED).integers(0, 256, (vgg.patch_input_px, vgg.patch_input_px, 3), dtype=np.uint8)
with torch.no_grad():
    noise_sat, _, _ = crop_border(vgg.model(to_batch(noise_rgb, DEVICE)), baseline_configs[VGG_BASELINE]["sat_feature_crop_pct"])
tuple(noise_sat.shape)

In [ ]:
noise_rows = []
for scale in SEARCH_SCALES:
    drone = default_correlators[VGG_BASELINE].drone_map(shape_frame, scale)
    mean_cosine = mean_cosine_raw(drone, noise_sat)
    cells = drone.shape[2] * drone.shape[3]
    noise_rows.append({"scale": scale, "drone_cells": cells, "max_mean_cosine": float(mean_cosine.max()),
                       "max_sum_cosine": float(mean_cosine.max()) * cells})
noise_table = pd.DataFrame(noise_rows)
mean_cosine_not_monotone = not bool(np.all(np.diff(noise_table["max_mean_cosine"]) > 0))
noise_table, mean_cosine_not_monotone

# (b) Identity

In [ ]:
class LabelCropCorrelator(PositionCorrelator):
    """The drone map is the label map cut at the frame's true pose, covering `true_scale` x the logged footprint.

    `drone_map(frame, scale)` resamples that crop to the footprint `scale` claims, as the real drone side is.
    """

    def __init__(self, features: FeatureExtractor, config: CorrelatorConfig, true_yaw_deg: dict[str, float],
                 true_scale: float) -> None:
        super().__init__(features, config)
        self.true_yaw_deg, self.true_scale = true_yaw_deg, true_scale

    def drone_map(self, frame: DroneFrame, scale: float = 1.0) -> torch.Tensor:
        key = (frame.name, scale)
        if key not in self._drone_maps:
            width_m, height_m = self.footprint_m(frame)
            crop = self.features.satellite_patch(frame.position, self.true_yaw_deg[frame.name],
                                                 width_m * self.true_scale, height_m * self.true_scale).cpu()
            size = (round(height_m * scale / self.features.match_gsd_m), round(width_m * scale / self.features.match_gsd_m))
            self._drone_maps[key] = F.interpolate(crop, size=size, mode="bilinear", align_corners=False)
        return self._drone_maps[key]

In [ ]:
frame_list = list(frames.values())
identity_frames = [frame_list[i] for i in np.linspace(0, len(frame_list) - 1, IDENTITY_FRAMES).round().astype(int)]
true_yaw_deg = {f.name: f.yaw_deg for f in identity_frames}
logged_frames = [replace(f, yaw_deg=f.yaw_deg + IDENTITY_HEADING_ERROR_DEG) for f in identity_frames]
[f.name for f in identity_frames]

In [ ]:
def identity_run(true_scale: float, search_scales: list[float]) -> pd.DataFrame:
    """One row per draw: the pose and position the search returns when the drone map is a label crop."""
    config = correlator_config({**baseline_configs[LABEL_BASELINE], **search_keys, "scales": search_scales,
                                "pose_schedule": IDENTITY_SCHEDULE, "fine_pass_top_k": IDENTITY_FINE_TOP_K})
    correlator = LabelCropCorrelator(features[LABEL_BASELINE], config, true_yaw_deg, true_scale)
    rows = []
    for draw in benchmark_draws(correlator, logged_frames, IDENTITY_NOISE_M, N_DIRECTIONS, SEED, progress=True):
        correction = decide(draw.surface, config)
        scored = [h for h in evaluate_schedule(correlator, draw.frame, draw.prior).values() if h is not None]
        true_pose = score_pose(correlator, draw.frame, draw.prior, -IDENTITY_HEADING_ERROR_DEG, true_scale)
        rivals = [h.peak for h in scored if not (np.isclose(h.heading_offset_deg, -IDENTITY_HEADING_ERROR_DEG)
                                                 and np.isclose(h.scale, true_scale))]
        rows.append({"frame": draw.frame.name, "bearing_deg": draw.bearing_deg,
                     "true_pose_evaluated": any(np.isclose(h.heading_offset_deg, -IDENTITY_HEADING_ERROR_DEG)
                                                and np.isclose(h.scale, true_scale) for h in scored),
                     "true_pose_peak": true_pose.peak, "winner_peak": max(h.peak for h in scored),
                     "best_rival_peak": max(rivals),
                     "heading_offset_deg": correction.heading_offset_deg, "scale": correction.scale,
                     "error_m": haversine_m(correction.position, draw.frame.position), "peak_z": correction.peak_z,
                     "hypotheses": draw.surface.hypotheses, "skipped": draw.surface.skipped_hypotheses})
    return pd.DataFrame(rows)

In [ ]:
schedule_name = IDENTITY_SCHEDULE if IDENTITY_SCHEDULE == "greedy" else f"{IDENTITY_SCHEDULE}, fine top {IDENTITY_FINE_TOP_K}"
identity_cases = {
    f"{schedule_name}: heading +{IDENTITY_HEADING_ERROR_DEG:g}, heading search": (1.0, [1.0]),
    f"{schedule_name}: heading +{IDENTITY_HEADING_ERROR_DEG:g}, heading and scale search": (1.0, search_keys["scales"]),
    f"{schedule_name}: heading +{IDENTITY_HEADING_ERROR_DEG:g}, scale {IDENTITY_SCALE:g}, heading and scale search": (IDENTITY_SCALE, search_keys["scales"]),
}
identity = {name: identity_run(true_scale, search_scales) for name, (true_scale, search_scales) in identity_cases.items()}
pd.concat(identity, names=["case"]).to_csv(experiment_dir(EXP) / "artifacts" / f"checks-identity-{IDENTITY_SCHEDULE}-top{IDENTITY_FINE_TOP_K}.csv")

In [ ]:
identity_rows = []
for name, result in identity.items():
    true_scale, search_scales = identity_cases[name]
    heading_error = (result["heading_offset_deg"] + IDENTITY_HEADING_ERROR_DEG).abs()
    recovered = ((heading_error <= IDENTITY_HEADING_TOLERANCE_DEG) & np.isclose(result["scale"], true_scale)
                 & (result["error_m"] < IDENTITY_POSITION_TOLERANCE_M))
    ambiguous = result["best_rival_peak"] >= result["true_pose_peak"]   # another pose outscores (or ties) the truth
    identity_rows.append({"case": name, "draws": len(result),
                          "heading_ok": int((heading_error <= IDENTITY_HEADING_TOLERANCE_DEG).sum()),
                          "scale_ok": int(np.isclose(result["scale"], true_scale).sum()),
                          "position_ok": int((result["error_m"] < IDENTITY_POSITION_TOLERANCE_M).sum()),
                          "max_heading_error_deg": float(heading_error.max()), "max_error_m": float(result["error_m"].max()),
                          "median_error_m": float(result["error_m"].median()), "skipped_hypotheses": int(result["skipped"].sum()),
                          "true_pose_not_evaluated": int((~result["true_pose_evaluated"]).sum()),
                          "true_pose_beats_winner": int((result["true_pose_peak"] > result["winner_peak"]).sum()),
                          "recovered": int(recovered.sum()), "map_ambiguity": int(ambiguous.sum()),
                          "required": int((~ambiguous).sum()), "required_recovered": int((recovered & ~ambiguous).sum()),
                          # amendment 2: scale cases need every non-ambiguous draw; heading-only cases need every draw
                          "pass": bool((recovered & ~ambiguous).sum() == (~ambiguous).sum()) if len(search_scales) > 1
                                  else bool(recovered.all())})
identity_table = pd.DataFrame(identity_rows)
identity_ok = bool(identity_table["pass"].all())
identity_table, identity_ok

# Plots

In [ ]:
fig_identity, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, result in identity.items():
    axes[0].plot(result["heading_offset_deg"].to_numpy(), "o", ms=4, label=name)
    axes[1].plot(result["error_m"].to_numpy(), "o", ms=4, label=name)
axes[0].axhline(-IDENTITY_HEADING_ERROR_DEG, color="k", ls=":", lw=1)
axes[0].set_xlabel("draw"); axes[0].set_ylabel("heading offset found, deg")
axes[1].axhline(IDENTITY_POSITION_TOLERANCE_M, color="k", ls=":", lw=1)
axes[1].set_xlabel("draw"); axes[1].set_ylabel("position error, m"); axes[1].legend(fontsize=7)
fig_identity.suptitle("identity: label crop at the true pose, logged heading off by +20 deg", fontsize=9)
fig_identity.tight_layout()

In [ ]:
fig_noise, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(noise_table["drone_cells"], noise_table["max_mean_cosine"], "o-", label="max mean cosine (raw score)")
ax2 = ax.twinx()
ax2.plot(noise_table["drone_cells"], noise_table["max_sum_cosine"], "s--", color="C1", label="max summed cosine")
ax.set_xlabel("drone feature-map cells (scale 0.5 to 1.0)"); ax.set_ylabel("max mean cosine"); ax2.set_ylabel("max summed cosine")
fig_noise.legend(fontsize=8, loc="upper center"); ax.set_title("VGG11 drone map against a noise satellite", fontsize=9)
fig_noise.tight_layout()

In [ ]:
figures = experiment_dir(EXP) / "figures"
figures.mkdir(exist_ok=True)
fig_identity.savefig(figures / f"checks-identity-{IDENTITY_SCHEDULE}-top{IDENTITY_FINE_TOP_K}.png", dpi=100)
fig_noise.savefig(figures / "checks-mean-cosine-noise.png", dpi=100)

In [ ]:
summary = {
    "no_regression": no_regression, "default_unchanged": default_unchanged, "search_path_equal": search_path_equal,
    "search_path_ties": ties.to_dict(orient="records"),
    "replay": replay_table.to_dict(orient="records"),
    "full_replay": full_replay.to_dict(orient="records"),
    "drone_map_shape_ok": bool(shapes["ok"].all()), "drone_map_shapes": shapes.astype(str).to_dict(orient="records"),
    "mean_cosine_not_monotone": mean_cosine_not_monotone, "mean_cosine_noise": noise_table.to_dict(orient="records"),
    "identity_schedule": IDENTITY_SCHEDULE, "identity_fine_top_k": IDENTITY_FINE_TOP_K, "identity_ok": identity_ok,
    "logged_run_replay": logged_replay.to_dict(orient="records"), "logged_runs_replay_ok": logged_runs_replay_ok, "identity": identity_table.to_dict(orient="records"),
}
(experiment_dir(EXP) / "artifacts" / CHECKS_OUT).write_text(json.dumps(summary, indent=1, default=float))

In [ ]:
assert no_regression, "default or search path moved a replayed draw"
assert summary["drone_map_shape_ok"], "scaled drone map is not scale x the full one"
assert mean_cosine_not_monotone, "mean cosine still grows with kernel size on noise"
assert identity_ok, "identity search missed the heading, scale or position"
assert logged_runs_replay_ok, "a logged 003 run no longer replays"